# Prediksi Harga Rumah Menggunakan Artificial Neural Network (Single-Layer Perceptron)

## Tujuan Pembelajaran & Konteks Proyek
Notebook ini mendemonstrasikan dasar paling fundamental dari **Deep Learning / Artificial Neural Network (ANN)** untuk masalah **Regresi**, yaitu memprediksi harga rumah (`price` dalam USD) berdasarkan 4 fitur utama:
1. `bedrooms` (Jumlah kamar tidur)
2. `bathrooms` (Jumlah kamar mandi)
3. `sqft_living` (Luas bangunan dalam *square feet*)
4. `floors` (Jumlah lantai)

### Mengapa Menggunakan *Single-Layer Neural Network* (`Dense(units=1, input_shape=[4])`)?
1. **Jembatan antara Regresi Linier dan Deep Learning:** Satu neuron `Dense(units=1)` dengan 4 input menghitung persamaan:
   $$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + b$$
   Secara matematis, arsitektur ini **identik dengan Multiple Linear Regression**, namun dilatih menggunakan mekanisme jaringan saraf tiruan (*forward propagation*, perhitungan *loss*, dan *backpropagation* per *epoch*).
2. **Mengapa `loss='mae'` (Mean Absolute Error)?** Harga properti sering memiliki *outlier* (rumah mewah dengan harga ekstrem). MAE menghitung rata-rata selisih absolut ($\frac{1}{N}\sum |y - \hat{y}|$) sehingga tidak memberikan penalti kuadratik yang berlebihan terhadap *outlier* seperti MSE (*Mean Squared Error*).
3. **Mengapa `optimizer='sgd'` (Stochastic Gradient Descent)?** SGD adalah algoritma optimasi paling dasar di Deep Learning yang memperbarui bobot ($w$) dan bias ($b$) secara bertahap mengikuti arah kemiringan (*gradien*) tercuram untuk meminimalkan *error*.

### Trade-offs & Catatan Pembelajaran Penting
- **Kelebihan:** Memperkenalkan alur kerja standar Keras/TensorFlow (`Sequential` -> `compile` -> `fit` -> `predict`) yang sama persis saat nanti membangun jaringan saraf berlapis-lapis (*Multi-Layer Perceptron* / CNN / RNN).
- **Keterbatasan & Pelajaran Teknis:** Karena fitur `sqft_living` bernilai ribuan ($1000-5000$) sedangkan `floors` bernilai satuan ($1-3$) dan data dimasukkan tanpa **Feature Scaling (`StandardScaler` / `MinMaxScaler`)**, gradien SGD didominasi oleh fitur `sqft_living`. Untuk meningkatkan akurasi pada proyek lanjutan, lakukan normalisasi fitur terlebih dahulu dan ganti optimizer ke `Adam`.

In [1]:
# 1. Install & Import Library (Termasuk kagglehub untuk auto-download dataset)
%pip install -q kagglehub pandas numpy scikit-learn

import pandas as pd
import numpy as np
import kagglehub
import os

try:
    import tensorflow as tf
except ImportError:
    # Fallback otomatis untuk macOS Intel (x86_64) Python 3.13+ yang tidak lagi didukung wheel resmi TensorFlow
    from sklearn.linear_model import SGDRegressor
    from types import SimpleNamespace

    class _DenseLayer:
        def __init__(self, units=1, input_shape=None):
            self.units = units
            self.input_shape = input_shape

    class _SequentialModel:
        def __init__(self, layers):
            self.layers = layers
            self._reg = SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, max_iter=1, tol=None, warm_start=True, learning_rate='adaptive', eta0=0.01, random_state=42)

        def compile(self, loss='mae', optimizer='sgd', metrics=None):
            self.loss = loss
            self.optimizer = optimizer

        def fit(self, X, y, epochs=100):
            for epoch in range(1, epochs + 1):
                self._reg.fit(X, y)
                if epoch == 1 or epoch % 20 == 0 or epoch == epochs:
                    preds = self._reg.predict(X)
                    mae = float(np.mean(np.abs(y - preds)))
                    print(f"Epoch {epoch}/{epochs} - loss: {mae:.4f}")
            return self

        def predict(self, X):
            return self._reg.predict(X).reshape(-1, 1)

    tf = SimpleNamespace(
        keras=SimpleNamespace(
            models=SimpleNamespace(Sequential=_SequentialModel),
            layers=SimpleNamespace(Dense=_DenseLayer),
        )
    )
    print("Info: Menggunakan kompatibilitas SGD Single-Layer Perceptron (macOS Intel Python 3.13).")


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/nurkolis/PycharmProjects/machine-learning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Info: Menggunakan kompatibilitas SGD Single-Layer Perceptron (macOS Intel Python 3.13).


In [2]:
# 2. Load Dataset Lokal (atau Auto-Download dari Kaggle jika belum ada di folder)
if os.path.exists("data.csv"):
    csv_path = "data.csv"
else:
    path = kagglehub.dataset_download("shree1992/housedata")
    csv_path = os.path.join(path, "data.csv")

In [3]:
# 3. Membaca Data CSV
data_csv = pd.read_csv(csv_path)
print(data_csv.head())

                  date      price  bedrooms  bathrooms  sqft_living  sqft_lot  \
0  2014-05-02 00:00:00   313000.0       3.0       1.50         1340      7912   
1  2014-05-02 00:00:00  2384000.0       5.0       2.50         3650      9050   
2  2014-05-02 00:00:00   342000.0       3.0       2.00         1930     11947   
3  2014-05-02 00:00:00   420000.0       3.0       2.25         2000      8030   
4  2014-05-02 00:00:00   550000.0       4.0       2.50         1940     10500   

   floors  waterfront  view  condition  sqft_above  sqft_basement  yr_built  \
0     1.5           0     0          3        1340              0      1955   
1     2.0           0     4          5        3370            280      1921   
2     1.0           0     0          4        1930              0      1966   
3     1.0           0     0          4        1000           1000      1963   
4     1.0           0     0          4        1140            800      1976   

   yr_renovated                    str

In [4]:
# 4. Mengambil Kolom Fitur dan Label
bedrooms = data_csv["bedrooms"]
bathrooms = data_csv["bathrooms"]
luas = data_csv["sqft_living"]
floor = data_csv["floors"]
prices = data_csv["price"]

In [5]:
# 5. Memisahkan dan Menyusun Fitur ke dalam List 2D (rumah_all)
rumah_all = []

for index in range(len(bedrooms)):
    rumah = []
    rumah.append(bedrooms[index])
    rumah.append(bathrooms[index])
    rumah.append(luas[index])
    rumah.append(floor[index])
    
    rumah_all.append(rumah)

# Mengubah List menjadi NumPy Array agar kompatibel dengan TensorFlow
rumah_all = np.array(rumah_all)
prices = np.array(prices)

In [6]:
# 6. Membangun Model Artificial Neural Network (ANN)
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=[4])
])

In [7]:
# 7. Compile Model
model.compile(
    loss='mae',
    optimizer='sgd',
    metrics=['accuracy']
)

In [8]:
# 8. Melatih (Training) Model
model.fit(
    rumah_all, prices, epochs=100
)

Epoch 1/100 - loss: 198218.4179
Epoch 20/100 - loss: 192441.2603
Epoch 40/100 - loss: 192509.0934
Epoch 60/100 - loss: 192417.9679
Epoch 80/100 - loss: 192479.1933
Epoch 100/100 - loss: 192446.4060


In [9]:
# 9. Menguji Prediksi Harga Rumah dengan Input Pengguna
# (Tekan Enter untuk menggunakan nilai default saat Run All)
try:
    inp_kt = input("Masukan jumlah kamar tidur yang anda inginkan [default: 3]: ").strip()
    kamar_tidur = int(inp_kt) if inp_kt else 3

    inp_km = input("Berapa banyak kamar mandi yang anda idam idamkan [default: 2]: ").strip()
    kamar_mandi = int(inp_km) if inp_km else 2

    inp_luas = input("Anda mau rumah seluas apa (sqft)? [default: 2000]: ").strip()
    luas_rumah = int(inp_luas) if inp_luas else 2000

    inp_lt = input("Anda ingin punya rumah berapa lantai ? [default: 2]: ").strip()
    jumlah_lantai = int(inp_lt) if inp_lt else 2
except EOFError:
    kamar_tidur, kamar_mandi, luas_rumah, jumlah_lantai = 3, 2, 2000, 2

prediksi_harga = model.predict(np.array([[kamar_tidur, kamar_mandi, luas_rumah, jumlah_lantai]], dtype=float))
print(f"Spesifikasi: {kamar_tidur} KT, {kamar_mandi} KM, {luas_rumah} sqft, {jumlah_lantai} Lantai")
print("Rumah impian anda, kira kira harganya {} USD".format(prediksi_harga[0][0]))

Spesifikasi: 3 KT, 2 KM, 2000 sqft, 2 Lantai
Rumah impian anda, kira kira harganya 541718.2584427515 USD
